In [ ]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns
from pathlib import Path


In [ ]:
import matplotlib as mpl

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

def fig_textwidth(height_ratio=0.62):
    import matplotlib.pyplot as plt
    return plt.subplots(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio))

In [ ]:
setup_pub_style(fontsize=9)

In [ ]:
def add_buoy_SAR_dt(df):    
    df["t0"] = pd.to_datetime(df["t0"])
    df["t1"] = pd.to_datetime(df["t1"])

    def extract_sar_time(path):
        filename = Path(path).stem  
        return pd.to_datetime(filename, format="%Y%m%dT%H%M")

    df["sar_t0"] = df["tiff0_path"].apply(extract_sar_time)
    df["sar_t1"] = df["tiff1_path"].apply(extract_sar_time)

    df["buoy_SAR_dt0_hours"] = (
        (df["t0"] - df["sar_t0"]).dt.total_seconds() / 3600
    )

    df["buoy_SAR_dt1_hours"] = (
        (df["t1"] - df["sar_t1"]).dt.total_seconds() / 3600
    )

    df["buoy_SAR_dt0_abs_hours"] = df["buoy_SAR_dt0_hours"].abs()
    df["buoy_SAR_dt1_abs_hours"] = df["buoy_SAR_dt1_hours"].abs()
    return df

In [ ]:
df_HV_HH = pd.read_csv('../../../buoy_dataset/IABP_validation/valid_HV_HH_24h_tol2_drift_pairs_template_25.csv')
df_HV = pd.read_csv('../../../buoy_dataset/IABP_validation/valid_HV_24h_tol2_drift_pairs_template_25.csv')
df_HH = pd.read_csv('../../../buoy_dataset/IABP_validation/valid_HH_24h_tol2_drift_pairs_template_25.csv')

In [ ]:
df_HV_HH.columns

In [ ]:
pos_count = (df_HV_HH['buoy_dx_m'] > 0).sum()
neg_count = (df_HV_HH['buoy_dx_m'] < 0).sum()
zero_count = (df_HV_HH['buoy_dx_m'] == 0).sum()

print(f"Positive: {pos_count}")
print(f"Negative: {neg_count}")
print(f"Zero: {zero_count}")

In [ ]:
pos_count = (df_HV_HH['buoy_dy_m'] > 0).sum()
neg_count = (df_HV_HH['buoy_dy_m'] < 0).sum()
zero_count = (df_HV_HH['buoy_dy_m'] == 0).sum()

print(f"Positive: {pos_count}")
print(f"Negative: {neg_count}")
print(f"Zero: {zero_count}")

In [ ]:
df_HV_HH = add_buoy_SAR_dt(df_HV_HH)
df_HV = add_buoy_SAR_dt(df_HV)
df_HH = add_buoy_SAR_dt(df_HH)

In [ ]:
# thresholds
df_HV_HH = df_HV_HH[(df_HV_HH["buoy_SAR_dt0_abs_hours"] <= 0.5) & (df_HV_HH["buoy_SAR_dt1_abs_hours"] <= 0.5)]
df_HV_HH = df_HV_HH[df_HV_HH['n_pm']>400]
df_HV_HH_no_outlier = df_HV_HH[(df_HV_HH['buoy_dy_m'] - df_HV_HH['sar_dy_m'])<8000]


df_HV = df_HV[(df_HV["buoy_SAR_dt0_abs_hours"] <= 0.5) & (df_HV["buoy_SAR_dt1_abs_hours"] <= 0.5)]
df_HV = df_HV[df_HV['n_pm']>400]

df_HH = df_HH[(df_HH["buoy_SAR_dt0_abs_hours"] <= 0.5) & (df_HH["buoy_SAR_dt1_abs_hours"] <= 0.5)]
df_HH = df_HH[df_HH['n_pm']>400]




In [ ]:
print(f"HV_HH val samples: {df_HV_HH['rel_endpoint_err'].count()}")
print(f"HV val samples: {df_HV['rel_endpoint_err'].count()}")
print(f"HH val samples: {df_HH['rel_endpoint_err'].count()}")


In [ ]:
import numpy as np

key_cols = ["tiff0_path", "tiff1_path"]
req_cols = ["angle_error_deg", "endpoint_err_m", "rel_endpoint_err"]

def _valid_rows(df):
    return (
        df.replace([np.inf, -np.inf], np.nan)
          .dropna(subset=key_cols + req_cols)
    )

# keep only rows with valid metrics in each df
df_HV_HH_v = _valid_rows(df_HV_HH)
df_HV_v    = _valid_rows(df_HV)
df_HH_v    = _valid_rows(df_HH)

# intersection of SAR pairs across all three (valid-only)
common_pairs = (
    df_HV_HH_v[key_cols]
    .merge(df_HV_v[key_cols], on=key_cols)
    .merge(df_HH_v[key_cols], on=key_cols)
    .drop_duplicates()
)

# filter each dataframe to only keep common, valid rows
df_HV_HH = df_HV_HH_v.merge(common_pairs, on=key_cols)
df_HV    = df_HV_v.merge(common_pairs, on=key_cols)
df_HH    = df_HH_v.merge(common_pairs, on=key_cols)



In [ ]:
# df_HV_HH_no_outlier['rel_endpoint_err'].describe()

In [ ]:
df_HV

In [ ]:
df_HV['rel_endpoint_err'].describe()


In [ ]:
df_HH['rel_endpoint_err'].describe()


In [ ]:
df_HV_HH_no_outlier["angle_error_deg"]

In [ ]:
common_tiff0_paths = (
    set(df_HH["tiff0_path"])
    & set(df_HV["tiff0_path"])
    & set(df_HV_HH_no_outlier["tiff0_path"])
)

df_HH_common = df_HH[df_HH["tiff0_path"].isin(common_tiff0_paths)].copy()
df_HV_common = df_HV[df_HV["tiff0_path"].isin(common_tiff0_paths)].copy()
df_HV_HH_common = df_HV_HH_no_outlier[df_HV_HH_no_outlier["tiff0_path"].isin(common_tiff0_paths)].copy()

In [ ]:
print(f"HV_HH val samples: {df_HV_HH_common['rel_endpoint_err'].count()}")
print(f"HV val samples: {df_HV_common['rel_endpoint_err'].count()}")
print(f"HH val samples: {df_HH_common['rel_endpoint_err'].count()}")

In [ ]:
def summarize(df):
    return {
        "n": len(df),
        "angle_mean_deg": df["angle_error_deg"].mean(),
        "angle_median_deg": df["angle_error_deg"].median(),
        "angle_std_deg": df["angle_error_deg"].std(),

        "abs_endpoint_mean_m": df["endpoint_err_m"].mean(),
        "abs_endpoint_median_m": df["endpoint_err_m"].median(),
        "abs_endpoint_std_m": df["endpoint_err_m"].std(),

        "rel_endpoint_mean": df["rel_endpoint_err"].mean(),
        "rel_endpoint_median": df["rel_endpoint_err"].median(),
        "rel_endpoint_std": df["rel_endpoint_err"].std(),

    }

summary = {
    "HH": summarize(df_HH),
    "HV": summarize(df_HV),
    "HH+HV": summarize(df_HV_HH_no_outlier),
}

import pandas as pd
summary_df = pd.DataFrame(summary).T
print(summary_df)


In [ ]:
def summarize(df):
    angle_abs = df["angle_error_deg"].abs()
    endpoint_abs = df["endpoint_err_m"].abs()
    rel_endpoint_abs = df["rel_endpoint_err"].abs()

    return {
        "n": len(df),

        "angle_mae_deg": angle_abs.mean(),
        "angle_median_abs_deg": angle_abs.median(),
        "angle_abs_std_deg": angle_abs.std(),

        "endpoint_mae_m": endpoint_abs.mean(),
        "endpoint_median_abs_m": endpoint_abs.median(),
        "endpoint_abs_std_m": endpoint_abs.std(),

        "rel_endpoint_mae": rel_endpoint_abs.mean(),
        "rel_endpoint_median_abs": rel_endpoint_abs.median(),
        "rel_endpoint_abs_std": rel_endpoint_abs.std(),
    }


summary = {
    "HH": summarize(df_HH_common),
    "HV": summarize(df_HV_common),
    "HH+HV": summarize(df_HV_HH_common),
}

import pandas as pd

summary_df = pd.DataFrame(summary).T
print(summary_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
import matplotlib as mpl
import matplotlib.ticker as mticker

# ---------- LaTeX / publication style ----------
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

sns.set_style("whitegrid")
sns.set_palette("colorblind")

KM = 1e-3  # meters -> kilometers

# ---------------- split inliers/outlier ----------------
threshold = 8000  # (buoy_dy_m - sar_dy_m) >= threshold is the outlier(s)

mask_outlier = (df_HV_HH["buoy_dy_m"] - df_HV_HH["sar_dy_m"]) >= threshold
df_in  = df_HV_HH.loc[~mask_outlier].copy()   # used for stats
df_out = df_HV_HH.loc[ mask_outlier].copy()   # plotted in orange only

print(f"Inliers: {len(df_in)}")
print(f"Outliers (orange): {len(df_out)}")


def scatter_with_1to1_km(ax, x_m, y_m, xlabel, ylabel, title):
    # correlation on inliers only (scale doesn't affect r)
    r, _ = pearsonr(df_in[x_m].to_numpy(), df_in[y_m].to_numpy())

    # inliers (km axes)
    sns.scatterplot(
        x=df_in[x_m].to_numpy() * KM,
        y=df_in[y_m].to_numpy() * KM,
        ax=ax, s=18, alpha=0.7, edgecolor="black"
    )

    # outlier(s) in orange (km axes)
    if len(df_out) > 0:
        sns.scatterplot(
            x=df_out[x_m].to_numpy() * KM,
            y=df_out[y_m].to_numpy() * KM,
            ax=ax, s=18, color="#d62728",
            edgecolor="black", linewidth=0.7, zorder=5
        )

    # 1:1 line + equal limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    lo = min(xlim[0], ylim[0])
    hi = max(xlim[1], ylim[1])
    ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.text(
        0.05, 0.95, f"$r$ = {r:.3f}",
        transform=ax.transAxes, va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85,
                  linewidth=0.5, edgecolor="0.7")
    )


# =========================
# Figure 1: TOP ROW (dx, dy)
# =========================
fig_top, axes_top = plt.subplots(
    1, 2, figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.48), dpi=300
)

scatter_with_1to1_km(
    axes_top[0],
    x_m="buoy_dx_m", y_m="sar_dx_m",
    xlabel="Buoy $dx$ (km)", ylabel="SAR $dx$ (km)",
    title="(a) x-component"
)

scatter_with_1to1_km(
    axes_top[1],
    x_m="buoy_dy_m", y_m="sar_dy_m",
    xlabel="Buoy $dy$ (km)", ylabel="SAR $dy$ (km)",
    title="(b) y-component"
)

sns.despine(fig=fig_top)
fig_top.tight_layout()
fig_top.savefig("SAR_Buoy_validation_HV_HH_top.pdf", bbox_inches="tight", dpi=300)
plt.show()


# ==================================
# Figure 2: BOTTOM ROW (errors + hist)
# ==================================
fig_bot, axes_bot = plt.subplots(
    1, 2, figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.48), dpi=300
)

# --- Left: component errors (inliers for stats; plot outlier in orange) ---
err_dx_m = (df_in["buoy_dx_m"] - df_in["sar_dx_m"]).to_numpy()
err_dy_m = (df_in["buoy_dy_m"] - df_in["sar_dy_m"]).to_numpy()

mean_x_m, mean_y_m = np.nanmean(err_dx_m), np.nanmean(err_dy_m)
std_x_m,  std_y_m  = np.nanstd(err_dx_m),  np.nanstd(err_dy_m)

sns.scatterplot(
    x=err_dx_m * KM, y=err_dy_m * KM,
    ax=axes_bot[0], s=18, alpha=0.7, edgecolor="black"
)

# outlier error point(s)
if len(df_out) > 0:
    err_dx_out = (df_out["buoy_dx_m"] - df_out["sar_dx_m"]).to_numpy()
    err_dy_out = (df_out["buoy_dy_m"] - df_out["sar_dy_m"]).to_numpy()
    sns.scatterplot(
        x=err_dx_out * KM, y=err_dy_out * KM,
        ax=axes_bot[0], s=18, color="#d62728",
        edgecolor="black", linewidth=0.7, zorder=5
    )

axes_bot[0].axhline(0, color="gray", linewidth=1)
axes_bot[0].axvline(0, color="gray", linewidth=1)
axes_bot[0].axhline(mean_y_m * KM, color="gray", linestyle="--", linewidth=1)
axes_bot[0].axvline(mean_x_m * KM, color="gray", linestyle="--", linewidth=1)

axes_bot[0].set_xlabel("$dx$ error (km)")
axes_bot[0].set_ylabel("$dy$ error (km)")
axes_bot[0].set_title("(a) Component errors (Buoy − SAR)")

axes_bot[0].text(
    0.05, 0.95,
    f"$\\mu_x$ = {mean_x_m:.1f} m, $\\sigma_x$ = {std_x_m:.1f} m\n"
    f"$\\mu_y$ = {mean_y_m:.1f} m, $\\sigma_y$ = {std_y_m:.1f} m",
    transform=axes_bot[0].transAxes, va="top",
    fontsize=8.5,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9,
              linewidth=0.5, edgecolor="0.7")
)

# --- Right: relative endpoint error histogram (log x), inliers only ---
rel_err = (
    df_in["rel_endpoint_err"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .to_numpy()
)
rel_err = rel_err[rel_err > 0]

mean_rel = np.mean(rel_err)
std_rel = np.std(rel_err)
n_rel = len(rel_err)

nbins = 30
bins = np.logspace(np.log10(rel_err.min()), np.log10(rel_err.max()), nbins + 1)

sns.histplot(rel_err, bins=bins, ax=axes_bot[1], edgecolor="0.3", linewidth=0.5)
axes_bot[1].set_xscale("log")

axes_bot[1].xaxis.set_major_formatter(mticker.ScalarFormatter())
axes_bot[1].xaxis.set_minor_formatter(mticker.NullFormatter())
axes_bot[1].ticklabel_format(style="plain", axis="x")

axes_bot[1].set_xlabel("Relative endpoint error (log scale)")
axes_bot[1].set_ylabel("Count")
axes_bot[1].set_title("(b)")

axes_bot[1].text(
    0.05, 0.95,
    f"$\\mu$ = {mean_rel:.3f}\n"
    f"$\\sigma$ = {std_rel:.3f}\n"
    f"$n$ = {n_rel}",
    transform=axes_bot[1].transAxes, va="top",
    fontsize=8.5,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9,
              linewidth=0.5, edgecolor="0.7")
)

sns.despine(fig=fig_bot)
fig_bot.tight_layout()
fig_bot.savefig("SAR_Buoy_validation_HV_HH_bottom.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr, lognorm
import matplotlib as mpl
import matplotlib.ticker as mticker

# ---------- LaTeX / publication style ----------
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

sns.set_style("whitegrid")
sns.set_palette("colorblind")

KM = 1e-3  # meters -> kilometers

# ---------------- split inliers/outlier ----------------
threshold = 8000  # (buoy_dy_m - sar_dy_m) >= threshold is the outlier(s)

mask_outlier = (df_HV_HH["buoy_dy_m"] - df_HV_HH["sar_dy_m"]) >= threshold
df_in = df_HV_HH.loc[~mask_outlier].copy()   # used for stats
df_out = df_HV_HH.loc[mask_outlier].copy()   # plotted in orange only

print(f"Inliers: {len(df_in)}")
print(f"Outliers (orange): {len(df_out)}")


def scatter_with_1to1_km(ax, x_m, y_m, xlabel, ylabel, title):
    # correlation on inliers only
    r, _ = pearsonr(df_in[x_m].to_numpy(), df_in[y_m].to_numpy())

    # inliers
    sns.scatterplot(
        x=df_in[x_m].to_numpy() * KM,
        y=df_in[y_m].to_numpy() * KM,
        ax=ax, s=18, alpha=0.7, edgecolor="black"
    )

    # outliers
    if len(df_out) > 0:
        sns.scatterplot(
            x=df_out[x_m].to_numpy() * KM,
            y=df_out[y_m].to_numpy() * KM,
            ax=ax, s=18, color="#d62728",
            edgecolor="black", linewidth=0.7, zorder=5
        )

    # 1:1 line + equal limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    lo = min(xlim[0], ylim[0])
    hi = max(xlim[1], ylim[1])
    ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.text(
        0.05, 0.95, f"$r$ = {r:.3f}",
        transform=ax.transAxes, va="top",
        fontsize=9,
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.85,
            linewidth=0.5,
            edgecolor="0.7",
        ),
    )


# ==================================
# Single 2x2 figure
# ==================================
fig, axes = plt.subplots(
    2, 2,
    figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.95),
    dpi=300
)

# ---- Top-left: dx comparison ----
scatter_with_1to1_km(
    axes[0, 0],
    x_m="buoy_dx_m", y_m="sar_dx_m",
    xlabel="Buoy $dx$ (km)", ylabel="SAR $dx$ (km)",
    title="(a) x-component"
)

# ---- Top-right: dy comparison ----
scatter_with_1to1_km(
    axes[0, 1],
    x_m="buoy_dy_m", y_m="sar_dy_m",
    xlabel="Buoy $dy$ (km)", ylabel="SAR $dy$ (km)",
    title="(b) y-component"
)

# ---- Bottom-left: component errors ----
err_dx_m = (df_in["buoy_dx_m"] - df_in["sar_dx_m"]).to_numpy()
err_dy_m = (df_in["buoy_dy_m"] - df_in["sar_dy_m"]).to_numpy()

mean_x_m = np.nanmean(err_dx_m)
mean_y_m = np.nanmean(err_dy_m)
std_x_m = np.nanstd(err_dx_m)
std_y_m = np.nanstd(err_dy_m)

sns.scatterplot(
    x=err_dx_m * KM,
    y=err_dy_m * KM,
    ax=axes[1, 0],
    s=18,
    alpha=0.7,
    edgecolor="black"
)

if len(df_out) > 0:
    err_dx_out = (df_out["buoy_dx_m"] - df_out["sar_dx_m"]).to_numpy()
    err_dy_out = (df_out["buoy_dy_m"] - df_out["sar_dy_m"]).to_numpy()
    sns.scatterplot(
        x=err_dx_out * KM,
        y=err_dy_out * KM,
        ax=axes[1, 0],
        s=18,
        color="#d62728",
        edgecolor="black",
        linewidth=0.7,
        zorder=5
    )

axes[1, 0].axhline(0, color="gray", linewidth=1)
axes[1, 0].axvline(0, color="gray", linewidth=1)
axes[1, 0].axhline(mean_y_m * KM, color="gray", linestyle="--", linewidth=1)
axes[1, 0].axvline(mean_x_m * KM, color="gray", linestyle="--", linewidth=1)

axes[1, 0].set_xlabel("$dx$ error (km)")
axes[1, 0].set_ylabel("$dy$ error (km)")
axes[1, 0].set_title("(c) Component errors (Buoy − SAR)")

axes[1, 0].text(
    0.05, 0.95,
    f"$\\mu_x$ = {mean_x_m:.1f} m, $\\sigma_x$ = {std_x_m:.1f} m\n"
    f"$\\mu_y$ = {mean_y_m:.1f} m, $\\sigma_y$ = {std_y_m:.1f} m",
    transform=axes[1, 0].transAxes,
    va="top",
    fontsize=8.5,
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.9,
        linewidth=0.5,
        edgecolor="0.7",
    ),
)

# ---- Bottom-right: relative endpoint error histogram + lognormal overlay ----
rel_err = (
    df_in["rel_endpoint_err"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .to_numpy()
)
rel_err = rel_err[rel_err > 0]

mean_rel = np.mean(rel_err)
std_rel = np.std(rel_err)
n_rel = len(rel_err)

nbins = 30
bins = np.logspace(np.log10(rel_err.min()), np.log10(rel_err.max()), nbins + 1)

# histogram
sns.histplot(
    rel_err,
    bins=bins,
    ax=axes[1, 1],
    edgecolor="0.3",
    linewidth=0.5
)

# fit lognormal with loc fixed at 0
shape, loc, scale = lognorm.fit(rel_err, floc=0)

# expected counts in each log-spaced bin
bin_centers = np.sqrt(bins[:-1] * bins[1:])  # geometric centers
expected_counts = n_rel * (
    lognorm.cdf(bins[1:], s=shape, loc=loc, scale=scale)
    - lognorm.cdf(bins[:-1], s=shape, loc=loc, scale=scale)
)

axes[1, 1].plot(
    bin_centers,
    expected_counts,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="Lognormal fit"
)

axes[1, 1].set_xscale("log")
axes[1, 1].xaxis.set_major_formatter(mticker.ScalarFormatter())
axes[1, 1].xaxis.set_minor_formatter(mticker.NullFormatter())
axes[1, 1].ticklabel_format(style="plain", axis="x")

axes[1, 1].set_xlabel("Relative endpoint error (log scale)")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("(d) Error distribution")

axes[1, 1].text(
    0.05, 0.95,
    f"$\\mu$ = {mean_rel:.3f}\n"
    f"$\\sigma$ = {std_rel:.3f}\n"
    f"$n$ = {n_rel}",
    transform=axes[1, 1].transAxes,
    va="top",
    fontsize=8.5,
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.9,
        linewidth=0.5,
        edgecolor="0.7",
    ),
)

axes[1, 1].legend(frameon=True, fontsize=8)

sns.despine(fig=fig)
fig.tight_layout()
fig.savefig("SAR_Buoy_validation_HV_HH_2x2_lognormal.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
import numpy as np

def bootstrap_ci(x, stat=np.median, n=5000, alpha=0.05):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    boots = [stat(np.random.choice(x, size=len(x), replace=True)) for _ in range(n)]
    lo = np.percentile(boots, 100*(alpha/2))
    hi = np.percentile(boots, 100*(1-alpha/2))
    return stat(x), lo, hi

val, lo, hi = bootstrap_ci(df_HV_HH["rel_endpoint_err"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HV["rel_endpoint_err"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HH["rel_endpoint_err"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")

val, lo, hi = bootstrap_ci(df_HV_HH["endpoint_err_m"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HV["endpoint_err_m"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HH["endpoint_err_m"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")

val, lo, hi = bootstrap_ci(df_HV_HH["angle_error_deg"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HV["angle_error_deg"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")
val, lo, hi = bootstrap_ci(df_HH["angle_error_deg"].values, stat=np.mean)
print(f"mean rel endpoint err: {val:.3f} (95% CI {lo:.3f}–{hi:.3f})")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Work on one dataset (they now contain identical cases)
df_time = df_HV_HH.copy()

# Extract date from tiff0_path
df_time["date"] = pd.to_datetime(
    df_time["tiff0_path"]
    .str.extract(r"/(\d{4})/(\d{2})/(\d{2})/")
    .agg("-".join, axis=1),
    format="%Y-%m-%d",
    errors="coerce",
)

df_time = df_time.dropna(subset=["date"])

# Aggregate by month
monthly_counts = (
    df_time
    .set_index("date")
    .resample("ME")
    .size()
)

sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(10, 4))

monthly_counts.plot(kind="bar", ax=ax, width=0.8)

ax.set_xlabel("")
ax.set_ylabel("Number of samples")
ax.set_title("Temporal distribution of validation samples (monthly)")

sns.despine(ax=ax)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D

# ---------------- LaTeX / publication style ----------------
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

# ---------------- Regions ----------------
regions = [
    "region-27_0-82_951-35_52-83_8",
    "region-27_2-82_348-35_0-83_2",
    "region-27_765-81_753-35_0-82_6",
    "region-33_0-82_95-40_715-83_903",
    "region-33_2-81_641-39_585-82_6",
    "region-33_07-82_34-40_1-83_297",
    "region-38_38-81_728-45_44-82_6",
    "region-38_38-82_95-46_0-83_913",
    "region-38_65-82_33-45_61-83_293",
]

def _parse_region_box(region_str):
    """
    Format you used: region-<lon1>-<lat1>-<lon2>-<lat2>, '_' is decimal point.
    Returns (lon1, lat1, lon2, lat2).
    """
    s = region_str.replace("region-", "")
    parts = s.split("-")
    if len(parts) != 4:
        raise ValueError(f"Unexpected region format: {region_str}")

    lon1 = float(parts[0].replace("_", "."))
    lat1 = float(parts[1].replace("_", "."))
    lon2 = float(parts[2].replace("_", "."))
    lat2 = float(parts[3].replace("_", "."))

    return lon1, lat1, lon2, lat2

def _month_to_season(m):
    return (
        "DJF" if m in (12, 1, 2) else
        "MAM" if m in (3, 4, 5) else
        "JJA" if m in (6, 7, 8) else
        "SON"
    )

# ---------------- Colors (colorblind palette, but avoid blue-on-blue) ----------------
sns.set_palette("colorblind")
pal = sns.color_palette("colorblind")
season_colors = {
    "DJF": pal[1],  # orange
    "MAM": pal[2],  # green
    "JJA": pal[4],  # purple (less blue)
    "SON": pal[6],  # pink
}

def plot_buoy_starts_with_region_boxes_by_season(
    df: pd.DataFrame,
    region_list,
    title="Buoy $t_0$ locations by season",
    save="val_buoy_locations_t0.pdf",
    height_ratio=0.70,
    pad_x_km=150,     # <-- widen map left/right in projected km
    pad_y_km=80,      # <-- widen map up/down in projected km
    ocean_color="#b0d4f1",
    land_color="0.92",
    box_color="black",
    box_lw=1.2,
    box_alpha=0.4,
    point_size=12,
    point_alpha=0.9,
    point_edgecolor="black",
    point_lw=0.4,
    markerscale=2.2,
):
    d = df.copy()

    # numeric coords
    d["t0_lat"] = pd.to_numeric(d["t0_lat"], errors="coerce")
    d["t0_lon"] = pd.to_numeric(d["t0_lon"], errors="coerce")

    # extract date from tiff0_path: .../YYYY/MM/DD/....
    d["date"] = pd.to_datetime(
        d["tiff0_path"].str.extract(r"/(\d{4})/(\d{2})/(\d{2})/").agg("-".join, axis=1),
        format="%Y-%m-%d",
        errors="coerce",
    )

    d = d.dropna(subset=["t0_lat", "t0_lon", "date"])

    d["season"] = d["date"].dt.month.map(_month_to_season)
    d["season"] = pd.Categorical(d["season"], categories=["DJF", "MAM", "JJA", "SON"], ordered=True)

    boxes = {r: _parse_region_box(r) for r in region_list}

    # ---- Build list of lon/lat points to define extent (starts + box corners) ----
    lonlat = []
    lonlat.extend(list(zip(d["t0_lon"].to_numpy(), d["t0_lat"].to_numpy())))
    for (lon1, lat1, lon2, lat2) in boxes.values():
        lonlat.extend([(lon1, lat1), (lon1, lat2), (lon2, lat1), (lon2, lat2)])

    lonlat = np.array(lonlat)
    lons = lonlat[:, 0]
    lats = lonlat[:, 1]

    # ---- Projection + transform to projected x/y (meters) ----
    proj = ccrs.NorthPolarStereo()
    pc = ccrs.PlateCarree()

    xy = proj.transform_points(pc, lons, lats)  # (N, 3) with x,y,z
    xs = xy[:, 0]
    ys = xy[:, 1]

    # extent in projected meters + padding
    pad_x = pad_x_km * 1000.0
    pad_y = pad_y_km * 1000.0
    x0, x1 = np.nanmin(xs) - pad_x, np.nanmax(xs) + pad_x
    y0, y1 = np.nanmin(ys) - pad_y, np.nanmax(ys) + pad_y

    # ---- Figure ----
    fig = plt.figure(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio), dpi=300)
    ax = fig.add_subplot(1, 1, 1, projection=proj)

    ax.spines["geo"].set_edgecolor("black")
    ax.spines["geo"].set_linewidth(0.8)

    # IMPORTANT: set extent in projection coordinates (meters)
    ax.set_extent([x0, x1, y0, y1], crs=proj)

    # background + land + coasts
    ax.set_facecolor(ocean_color)
    ax.add_feature(cfeature.LAND, facecolor=land_color, edgecolor="none", zorder=0)
    ax.coastlines(color="black", linewidth=0.9, zorder=2)

    # bounding boxes
    for name, (lon1, lat1, lon2, lat2) in boxes.items():
        xs_box = [lon1, lon2, lon2, lon1, lon1]
        ys_box = [lat1, lat1, lat2, lat2, lat1]
        ax.plot(
            xs_box, ys_box,
            transform=pc,
            color=box_color,
            linewidth=box_lw,
            alpha=box_alpha,
            zorder=3,
        )

    # points by season
    for season in ["DJF", "MAM", "JJA", "SON"]:
        sub = d[d["season"] == season]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["t0_lon"], sub["t0_lat"],
            s=point_size,
            facecolor=season_colors[season],
            edgecolor=point_edgecolor,
            linewidth=point_lw,
            transform=pc,
            label=f"{season} (n={len(sub)})",
            zorder=4,
            alpha=point_alpha,
        )


    # gl = ax.gridlines(draw_labels=False,
    #                 linewidth=0.5,
    #                 color="0.4",
    #                 alpha=0.6,
    #                 linestyle="--")
    # gl.bottom_labels = True 

    gl = ax.gridlines(draw_labels=True,
                    linewidth=0.5,
                    color="0.4",
                    alpha=0.6,
                    linestyle="--")
    gl.top_labels = False
    gl.left_labels = False
    gl.right_labels = False
    gl.bottom_labels = True

    # Set latitude gridline locations
    lat_ticks = np.array([81, 82, 83])
    gl.ylocator = mticker.FixedLocator(lat_ticks)

    lon_label = d["t0_lon"].min() + 26   

    for lat in lat_ticks:
        ax.text(
            lon_label, lat,
            f"{lat}°N",
            transform=ccrs.PlateCarree(),
            ha="right",
            va="center",
            fontsize=8,
            color="black",
            zorder=10,
            bbox=dict(
                boxstyle="round,pad=0.15",
                facecolor=None,
                edgecolor="0.7",
                alpha=0.0,
                linewidth=0.5,
            ),
        )

    # legend + total n
    handles, labels = ax.get_legend_handles_labels()
    total_n = len(d)
    handles.append(Line2D([], [], linestyle="none"))
    labels.append(f"Total n = {total_n}")

    ax.legend(
        handles, labels,
        frameon=True,
        facecolor="white",
        edgecolor="0.7",
        framealpha=0.95,
        loc="lower left",
        scatterpoints=1,
        markerscale=markerscale,
    )

    # ax.set_title(f"{total_n} {title}")

    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.05)

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)

    plt.show()
    return fig, ax

# ---- Usage ----
plot_buoy_starts_with_region_boxes_by_season(
    df_HV_HH,
    regions,
    title="",
    save="val_buoy_locations_t0.pdf",
    height_ratio=0.5,
    pad_x_km=180,   
    pad_y_km=15, 
    point_lw=0.5
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

regions = [
    "region-27_0-82_951-35_52-83_8",
    "region-27_2-82_348-35_0-83_2",
    "region-27_765-81_753-35_0-82_6",
    "region-33_0-82_95-40_715-83_903",
    "region-33_2-81_641-39_585-82_6",
    "region-33_07-82_34-40_1-83_297",
    "region-38_38-81_728-45_44-82_6",
    "region-38_38-82_95-46_0-83_913",
    "region-38_65-82_33-45_61-83_293",
]

def _parse_region_box(region_str):
    s = region_str.replace("region-", "")
    parts = s.split("-")
    if len(parts) != 4:
        raise ValueError(f"Unexpected region format: {region_str}")
    lon1 = float(parts[0].replace("_", "."))
    lat1 = float(parts[1].replace("_", "."))
    lon2 = float(parts[2].replace("_", "."))
    lat2 = float(parts[3].replace("_", "."))
    return lon1, lat1, lon2, lat2

def _month_to_season(m):
    return (
        "DJF" if m in (12, 1, 2) else
        "MAM" if m in (3, 4, 5) else
        "JJA" if m in (6, 7, 8) else
        "SON"
    )

sns.set_palette("colorblind")
pal = sns.color_palette("colorblind")
season_colors = {
    "DJF": pal[1],
    "MAM": pal[2],
    "JJA": pal[4],
    "SON": pal[6],
}

def plot_buoy_starts_with_region_boxes_by_season(
    df: pd.DataFrame,
    region_list,
    save="val_buoy_locations_t0.pdf",
    height_ratio=0.50,
    pad_x_km=180,
    pad_y_km=15,
    ocean_color="#b0d4f1",
    land_color="0.92",
    box_color="black",
    box_lw=1.2,
    box_alpha=0.4,
    point_size=12,
    point_alpha=0.9,
    point_edgecolor="black",
    point_lw=0.5,
    markerscale=2.2,
    lon_ticks=None,
    lat_ticks=None,
):
    d = df.copy()
    d["t0_lat"] = pd.to_numeric(d["t0_lat"], errors="coerce")
    d["t0_lon"] = pd.to_numeric(d["t0_lon"], errors="coerce")
    d["date"] = pd.to_datetime(
        d["tiff0_path"].str.extract(r"/(\d{4})/(\d{2})/(\d{2})/").agg("-".join, axis=1),
        format="%Y-%m-%d", errors="coerce",
    )
    d = d.dropna(subset=["t0_lat", "t0_lon", "date"])
    d["season"] = d["date"].dt.month.map(_month_to_season)
    d["season"] = pd.Categorical(d["season"], categories=["DJF", "MAM", "JJA", "SON"], ordered=True)

    boxes = {r: _parse_region_box(r) for r in region_list}

    # collect all lon/lat points to auto-compute extent
    lonlat = list(zip(d["t0_lon"].to_numpy(), d["t0_lat"].to_numpy()))
    for (lon1, lat1, lon2, lat2) in boxes.values():
        lonlat.extend([(lon1, lat1), (lon1, lat2), (lon2, lat1), (lon2, lat2)])
    lonlat = np.array(lonlat)

    proj = ccrs.NorthPolarStereo()
    pc = ccrs.PlateCarree()

    xy = proj.transform_points(pc, lonlat[:, 0], lonlat[:, 1])
    xs, ys = xy[:, 0], xy[:, 1]

    pad_x = pad_x_km * 1000.0
    pad_y = pad_y_km * 1000.0
    x0 = np.nanmin(xs) - pad_x
    x1 = np.nanmax(xs) + pad_x
    y0 = np.nanmin(ys) - pad_y
    y1 = np.nanmax(ys) + pad_y

    fig = plt.figure(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio), dpi=300)
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    ax.spines["geo"].set_edgecolor("black")
    ax.spines["geo"].set_linewidth(0.8)
    ax.set_extent([x0, x1, y0, y1], crs=proj)
    ax.set_facecolor(ocean_color)
    ax.add_feature(cfeature.LAND, facecolor=land_color, edgecolor="none", zorder=0)
    ax.coastlines(color="black", linewidth=0.9, zorder=2)

    # region boxes
    for name, (lon1, lat1, lon2, lat2) in boxes.items():
        ax.plot(
            [lon1, lon2, lon2, lon1, lon1],
            [lat1, lat1, lat2, lat2, lat1],
            transform=pc, color=box_color, linewidth=box_lw,
            alpha=box_alpha, zorder=3,
        )

    # scatter points by season
    for season in ["DJF", "MAM", "JJA", "SON"]:
        sub = d[d["season"] == season]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["t0_lon"], sub["t0_lat"],
            s=point_size, facecolor=season_colors[season],
            edgecolor=point_edgecolor, linewidth=point_lw,
            transform=pc, label=f"{season} (n={len(sub)})",
            zorder=4, alpha=point_alpha,
        )

    # gridlines (no auto-labels)
    if lat_ticks is None:
        lat_ticks = np.array([81, 82, 83])
    if lon_ticks is None:
        lon_ticks = np.arange(10, 60, 10)

    gl = ax.gridlines(draw_labels=False, linewidth=0.5, color="0.4", alpha=0.6, linestyle="--")
    gl.xlocator = mticker.FixedLocator(lon_ticks)
    gl.ylocator = mticker.FixedLocator(lat_ticks)

    # manual longitude labels: find where each meridian crosses y0 (bottom edge)
    lats_search = np.linspace(78.0, 85.5, 2000)
    for lon in lon_ticks:
        pts = proj.transform_points(pc, np.full_like(lats_search, float(lon)), lats_search)
        idx = np.argmin(np.abs(pts[:, 1] - y0))
        x_label = pts[idx, 0]
        if x0 <= x_label <= x1:
            ax.text(
                x_label + 10000, y0 + 15000,
                f"{int(lon)}°E",
                transform=proj,
                ha="center", va="top",
                fontsize=7, color="black",
                clip_on=False, zorder=11,
            )

    # manual latitude labels
    lon_label = float(d["t0_lon"].min()) + 26.0
    for lat in lat_ticks:
        ax.text(
            lon_label, lat-0.2, f"{lat}°N",
            transform=pc, ha="right", va="center",
            fontsize=7, color="black", zorder=10,
        )

    # legend
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Line2D([], [], linestyle="none"))
    labels.append(f"Total n = {len(d)}")
    ax.legend(
        handles, labels, frameon=True, facecolor="white",
        edgecolor="0.7", framealpha=0.95, loc="upper left",
        scatterpoints=1, markerscale=markerscale,
    )

    fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.08)
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)
    plt.show()
    return fig, ax

plot_buoy_starts_with_region_boxes_by_season(
    df_HV_HH,
    regions,
    save="val_buoy_locations_t0_new.pdf",
    height_ratio=0.4,
    pad_x_km=180,
    pad_y_km=2,
)